# Self-attention from scratch in PyTorch

*A companion to the blog post “Self-attention from scratch in PyTorch” (Part 5 of the Transformer on-ramp series).*

Build scaled dot-product attention, add a causal mask, draw it, and check it against PyTorch’s own implementation.

In [ ]:
import torch

def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V: (batch, heads, seq, d_k)
    scores = Q @ K.transpose(-2, -1) / (K.shape[-1] ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = scores.softmax(dim=-1)
    return weights @ V, weights

## Test it on random data

In [ ]:
torch.manual_seed(0)
B, H, T, dk = 2, 4, 5, 8
Q, K, V = (torch.randn(B, H, T, dk) for _ in range(3))
mask = torch.tril(torch.ones(T, T))                  # 1 where reading is allowed
out, w = scaled_dot_product_attention(Q, K, V, mask)

print('output shape      :', tuple(out.shape))
print('rows sum to 1     :', torch.allclose(w.sum(-1), torch.ones(B, H, T)))
print('no future leakage :', bool((w.triu(1) == 0).all()))
ref = torch.nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True)
print('matches PyTorch   :', torch.allclose(out, ref, atol=1e-5))

## See the causal mask in the weights

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(3.6, 3.2))
plt.imshow(w[0, 0].detach(), cmap='Purples')
plt.title('Attention weights, one head')
plt.xlabel('token being read'); plt.ylabel('token asking')
plt.colorbar(); plt.show()

The upper triangle is zero: each token reads only itself and earlier tokens.

## A single-head self-attention layer
In a real model Q, K and V come from learned projections of the token vectors:

In [ ]:
class SelfAttention(torch.nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.q = torch.nn.Linear(d_model, d_k, bias=False)
        self.k = torch.nn.Linear(d_model, d_k, bias=False)
        self.v = torch.nn.Linear(d_model, d_k, bias=False)
    def forward(self, x):                      # x: (batch, seq, d_model)
        T = x.shape[1]
        mask = torch.tril(torch.ones(T, T))
        out, w = scaled_dot_product_attention(self.q(x).unsqueeze(1), self.k(x).unsqueeze(1), self.v(x).unsqueeze(1), mask)
        return out.squeeze(1), w.squeeze(1)

layer = SelfAttention(d_model=16, d_k=8)
y, weights = layer(torch.randn(2, 5, 16))
print(y.shape, weights.shape)   # (2, 5, 8) (2, 5, 5)

**Try it:** remove the mask (pass `mask=None`) and re-run the plot. What changes?

### Want the full guided version?

This notebook is a small taster from **Chapter 1 of the course *Generative AI and LLMs with Python*** (course-2-generative-ai), one of 4 world-class courses on [aibits.ecubeanalytics.com](https://aibits.ecubeanalytics.com/?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01).

**The full guided lab, quizzes and projects are in the course.** Chapter 1 is part of the free first module:

- [Read Chapter 1 free (sign up)](https://aibits.ecubeanalytics.com/main/document/genai_gate.php?c=course-2-generative-ai&f=chapters/course-2-generative-ai/01-on-ramp-neurons-to-the-transformer.html&utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
- [See the course demo](https://aibits.ecubeanalytics.com/main/document/genai_public/GENAILLMPYTHON.html?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
